In [1]:
import pathlib
from collections import OrderedDict
from typing import Any, Iterable

import ray
import numpy as np
import anndata
import pandas as pd
# from bolero.tl.dataset.sc_transforms import FilterRegions
# from bolero.tl.dataset.transforms import (
#     FetchRegionOneHot,
#     ReverseComplement,
# )

# from bolero.utils import get_global_coords, understand_regions
# from bolero.tl.dataset.file_transforms import FetchRegionBigWigs, GetEmbedding
# from utils import BorzoiRegions, clamp_sqrt_large_value

# DNA_NAME = "dna_one_hot"
from bolero.tl.model.borzoi.dataset import BorzoiDatasetOnline
from bolero import init

init(num_cpus=32, object_store_memory_gb=100)

2024-10-07 16:27:00,799	INFO worker.py:1762 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


In [2]:
data_dir = '/large_storage/zhoulab/tlgallent/data/Li2023Science/'
bw_file = f'{data_dir}/bigwig/ASC.bw'
bed = f'{data_dir}peaks/cCREs.bed'


In [3]:
bigwig_paths = [bw_file]

In [4]:
adata = anndata.read_h5ad('/large_storage/zhoulab/tlgallent/data/cell_29000_rna_raw_gencode_adata_with_embeddings.h5ad')
scvi_embedding = adata.obsm['X_scVI']
# Create a DataFrame with embeddings and cell types
df = pd.DataFrame(scvi_embedding, index=adata.obs.index)
df['cell_type'] = adata.obs['MajorType']
# Group by cell type
grouped = df.groupby('cell_type').mean()
# Get embedding
recalculated_embedding = grouped.to_numpy()


leg_map = {item: index for index, item in enumerate(grouped.index.to_list())}
leg_map

/tmp/ipykernel_2685326/135511446.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby('cell_type').mean()


{'ASC': 0,
 'Amy': 1,
 'CHD7': 2,
 'EC': 3,
 'Foxp2': 4,
 'L4_IT': 5,
 'L5_ET': 6,
 'L5_IT': 7,
 'L6_CT': 8,
 'L6_IT': 9,
 'L6_IT_Car3': 10,
 'L6b': 11,
 'L23_IT': 12,
 'L56_NP': 13,
 'Lamp5': 14,
 'Lamp5_LHX6': 15,
 'MGC': 16,
 'MSN_D1': 17,
 'MSN_D2': 18,
 'ODC': 19,
 'OPC': 20,
 'PC': 21,
 'Pvalb': 22,
 'Pvalb_ChC': 23,
 'Sncg': 24,
 'Sst': 25,
 'SubCtx': 26,
 'VLMC': 27,
 'Vip': 28}

In [5]:
borzoi_online_dataset = BorzoiDatasetOnline(bigwig_paths=bigwig_paths, bed=bed, leg_map=leg_map, genome="hg38")

In [6]:
borzoi_online_dataset.train()

In [7]:
borzoi_online_dataset.get_default_config()

{'bigwig_paths': 'REQUIRED',
 'bed': 'REQUIRED',
 'leg_map': 'REQUIRED',
 'lora': 'REQUIRED',
 'genome': 'hg38',
 'batch_size': 2,
 'dna_window': 524288,
 'pos_resolution': 32,
 'reverse_complement': True,
 'max_jitter': 0,
 'clamp_sqrt_threshold': None,
 'shuffle_files': True,
 'read_parquet_kwargs': None}

In [8]:
vars(borzoi_online_dataset)

{'genome': Genome: hg38
 Fasta Path: /home/tlgallent/projects/software/bolero/src/bolero/data/hg38/fasta/hg38.fa
 Genome One Hot Zarr: Not created,
 'bed':        Chromosome     Start       End                  region  \
 0            chr1    796853   1321141     chr1:796853-1321141   
 1            chr1    797560   1321848     chr1:797560-1321848   
 2            chr1    801832   1326120     chr1:801832-1326120   
 3            chr1    804743   1329031     chr1:804743-1329031   
 4            chr1    806629   1330917     chr1:806629-1330917   
 ...           ...       ...       ...                     ...   
 502537       chrY  26098695  26622983  chrY:26098695-26622983   
 502538       chrY  26104161  26628449  chrY:26104161-26628449   
 502539       chrY  26108746  26633034  chrY:26108746-26633034   
 502540       chrY  26110462  26634750  chrY:26110462-26634750   
 502541       chrY  26110997  26635285  chrY:26110997-26635285   
 
                  Original_Name  
 0         chr1:1

In [9]:
loader1 = borzoi_online_dataset.get_dataloader(folds=0, region_bed=bed)

for batch in loader1:
    pass

for k, v in batch.items():
    print(k, v.dtype, v.shape)
del loader1

UnboundLocalError: local variable 'dataset' referenced before assignment